In [18]:
import sys
sys.path.insert(0, '..')

import polars as pl
from diaphanous.show import show

pl.Config.set_thousands_separator(",")

%load_ext rpy2.ipython
import diaphanous.arr as arr
arr.install()
RLIB = arr.RLIB

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
import rpy2.rinterface as ri

ro.globalenv['RLIB'] = RLIB
ro.r(".libPaths(RLIB)")

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [19]:
reports = pl.read_csv("../data/ocse-reports-per-year.csv")
population = pl.read_csv("../data/populations-simple.csv")
internet_users = pl.read_csv("../data/internet-users-un.csv")
social_media_accounts = pl.read_csv("../data/social-accounts.csv")

frame = reports.join(
    population, on="year", how="left"
).join(
    internet_users, on="year", how="left"
).join(
    social_media_accounts, on="year", how="left"
).select(
    pl.col("year", "reports", "population"),
    (pl.col("million_accounts") * 1_000_000).alias("accounts"),
    (pl.col("internet_users_pct") * pl.col("population") / 100).alias("internet_users"),
).filter(
    (pl.col("year") <= 2023) & (pl.col("year") >= 2014)
).drop_nans()

corrs = frame.corr().select(
    pl.col("internet_users", "population", "accounts")
).row(1)

data = frame.to_pandas()

show(data)

,year,reports,population,accounts,internet_users
0,"2,023","36,210,368","8,091,734,930.0","4,770,000,000.0","5,291,994,644.2"
1,"2,022","32,059,029","8,021,407,192.0","4,632,000,000.0","5,109,636,381.3"
2,"2,021","29,397,681","7,954,448,391.5","4,214,000,000.0","4,907,894,657.6"
3,"2,020","21,751,085","7,887,001,292.0","3,726,000,000.0","4,621,782,757.1"
4,"2,019","16,987,361","7,811,293,698.5","3,478,000,000.0","4,132,174,366.5"
5,"2,018","18,462,422","7,729,902,780.5","3,212,000,000.0","3,749,002,848.5"
6,"2,017","10,214,753","7,645,617,954.0","2,804,000,000.0","3,455,819,315.2"
7,"2,016","8,297,923","7,558,554,525.5","2,320,000,000.0","3,235,061,336.9"
8,"2,015","4,403,657","7,470,491,871.5","2,094,000,000.0","2,973,255,764.9"
9,"2,014","1,106,071","7,381,616,244.0","1,869,000,000.0","2,760,724,475.3"


In [20]:
stats = importr('stats')
base = importr('base')

with (ro.default_converter + pandas2ri.converter).context():
    rdata = ro.conversion.get_conversion().py2rpy(data)

ro.globalenv['data'] = rdata

ro.r("library(estimatr)")

r2_adj = {}
robust_r2_adj = {}

MODELS = [
    ('year', 'reports ~ year'),
    ('pop', 'reports ~ population'),
    ('inet', 'reports ~ internet_users'),
    ('social', 'reports ~ accounts'),
    ('year_pop', 'reports ~ year + population'),
    ('year_inet', 'reports ~ year + internet_users'),
    ('year_social', 'reports ~ year + accounts'),
    ('pop_inet', 'reports ~ population + internet_users'),
    ('pop_social', 'reports ~ population + accounts'),
    ('inet_social', 'reports ~ internet_users + accounts'),
    ('year_pop_inet', 'reports ~ year + population + internet_users'),
    ('year_pop_social', 'reports ~ year + population + accounts'),
    ('year_inet_social', 'reports ~ year + internet_users + accounts'),
    ('pop_inet_social', 'reports ~ population + internet_users + accounts'),
    ('year_pop_inet_social', 'reports ~ year + population + internet_users + accounts'),
]

for name, relation in MODELS:
    ro.r(f'mod_{name} <- lm({relation}, data = data)')
    r2_adj[name] = ro.r(f'summary(mod_{name})$adj.r.squared')
    ro.r(f'robust_mod_{name} <- lm_robust({relation}, data = data)')
    robust_r2_adj[name] = ro.r(f'summary(robust_mod_{name})$adj.r.squared')


In [21]:
show("<h2>Normality</h2>")

for name, _ in MODELS:
    print(ro.r(f'shapiro.test(rstandard(mod_{name}))'))

show("<h2>Adjusted R²</h2>")
for key, value in sorted(r2_adj.items(), key=lambda i: i[1][0]):
    print(f"{key}: {value}")

show("<h3>Best Linear Model: Social Media Accounts</h3>")
print(ro.r('mod_social'))
print(ro.r('summary(mod_social)'))

show("<h2>Robust Adjusted R²</h2>")
for key, value in sorted(robust_r2_adj.items(), key=lambda i: i[1][0]):
    print(f"{key}: {value}")

show("<h3>Best Linear Model: Social Media Accounts</h3>")
print(ro.r('robust_mod_social'))
print(ro.r('summary(robust_mod_social)'))


	Shapiro-Wilk normality test

data:  rstandard(mod_year)
W = 0.93124, p-value = 0.4602



	Shapiro-Wilk normality test

data:  rstandard(mod_pop)
W = 0.82879, p-value = 0.03236



	Shapiro-Wilk normality test

data:  rstandard(mod_inet)
W = 0.95526, p-value = 0.7307



	Shapiro-Wilk normality test

data:  rstandard(mod_social)
W = 0.92137, p-value = 0.3685



	Shapiro-Wilk normality test

data:  rstandard(mod_year_pop)
W = 0.90699, p-value = 0.261



	Shapiro-Wilk normality test

data:  rstandard(mod_year_inet)
W = 0.92608, p-value = 0.4104



	Shapiro-Wilk normality test

data:  rstandard(mod_year_social)
W = 0.92703, p-value = 0.4193



	Shapiro-Wilk normality test

data:  rstandard(mod_pop_inet)
W = 0.91565, p-value = 0.3221



	Shapiro-Wilk normality test

data:  rstandard(mod_pop_social)
W = 0.93679, p-value = 0.5179



	Shapiro-Wilk normality test

data:  rstandard(mod_inet_social)
W = 0.93385, p-value = 0.4868



	Shapiro-Wilk normality test

data:  rstandard(mod_year_pop_inet)

inet: [1] 0.9639632

pop_inet: [1] 0.9654064

pop: [1] 0.9664247

pop_inet_social: [1] 0.973231

year_inet: [1] 0.9739743

year_pop_inet_social: [1] 0.9741984

year_inet_social: [1] 0.9742663

year_pop_inet: [1] 0.9751935

pop_social: [1] 0.9768941

inet_social: [1] 0.9769869

year: [1] 0.9772185

year_social: [1] 0.9778598

year_pop_social: [1] 0.9781597

year_pop: [1] 0.9785615

social: [1] 0.9797739




Call:
lm(formula = reports ~ accounts, data = data)

Coefficients:
(Intercept)     accounts  
 -2.010e+07    1.147e-02  



Call:
lm(formula = reports ~ accounts, data = data)

Residuals:
     Min       1Q   Median       3Q      Max 
-2806881  -950845   125834  1487770  1786217 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.010e+07  1.896e+06   -10.6 5.48e-06 ***
accounts     1.147e-02  5.487e-04    20.9 2.88e-08 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1709000 on 8 degrees of freedom
Multiple R-squared:  0.982,	Adjusted R-squared:  0.9798 
F-statistic:   437 on 1 and 8 DF,  p-value: 2.878e-08




inet: [1] 0.9639632

pop_inet: [1] 0.9654064

pop: [1] 0.9664247

pop_inet_social: [1] 0.973231

year_inet: [1] 0.9739743

year_pop_inet_social: [1] 0.9741984

year_inet_social: [1] 0.9742663

year_pop_inet: [1] 0.9751935

pop_social: [1] 0.9768941

inet_social: [1] 0.9769869

year: [1] 0.9772185

year_social: [1] 0.9778598

year_pop_social: [1] 0.9781597

year_pop: [1] 0.9785615

social: [1] 0.9797739



                 Estimate   Std. Error   t value     Pr(>|t|)      CI Lower
(Intercept) -2.009925e+07 1.440713e+06 -13.95090 6.751501e-07 -2.342154e+07
accounts     1.147024e-02 4.324234e-04  26.52548 4.387673e-09  1.047307e-02
                 CI Upper DF
(Intercept) -1.677696e+07  8
accounts     1.246741e-02  8


Call:
lm_robust(formula = reports ~ accounts, data = data)

Standard error type:  HC2 

Coefficients:
              Estimate Std. Error t value  Pr(>|t|)   CI Lower   CI Upper DF
(Intercept) -2.010e+07  1.441e+06  -13.95 6.752e-07 -2.342e+07 -1.678e+07  8
accounts     1.147e-02  4.324e-04   26.53 4.388e-09  1.047e-02  1.247e-02  8

Multiple R-squared:  0.982 ,	Adjusted R-squared:  0.9798 
F-statistic: 703.6 on 1 and 8 DF,  p-value: 4.388e-09

